# UELer: Streaming from the BioImage Archive (BIA)

This notebook is the BIA counterpart of `run_ueler.ipynb`. Instead of pointing at a local
`base_folder`, it explores a **public BioImage Archive study** (`S-BIAD*`) directly over the
network — without downloading the whole dataset first.

### Instruction for Specifying the Study

BIA studies have no standard folder layout, so you configure the loader with:
  - **`source`**: a BIA accession id (e.g. `'S-BIAD2557'`, resolved via the BioStudies REST API) **or** a direct HTTPS base URL to the study.
  - **`descriptor`** (optional): a small mapping that tells UELer where FOVs / channels / masks live. It can be a `dict` or a path to a `.json` file. When omitted, UELer attempts to **auto-detect** the folder-per-FOV or OME-TIFF-per-FOV layouts.
  - **`local_dir`** (optional): the local workspace root. Defaults to `~/.ueler/bia/<accession>/`, which holds your persistent `.UELer` work (ROIs, checkpoints, palettes) plus a disposable `cache/` of downloaded images.

**Reads:** pyramidal OME-TIFFs are streamed via HTTP byte-range requests; other files (e.g. single-resolution MIBI TIFFs) are downloaded once into the cache and reused.

The descriptor below targets the reference dataset **`S-BIAD2557`** (a folder-per-FOV MIBI study). Adjust `source`/`descriptor` for a different study:

In [ ]:
# BIA study to explore
source = 'S-BIAD2557'  # accession id, or a direct HTTPS base URL to the study

# Optional layout descriptor (set to None to attempt auto-detection).
# For S-BIAD2557 (folder-per-FOV MIBI):
descriptor = {
    'mode': 'folder',
    'base': 'Files/spatial_murine_iCCAvsHCC/image_data',
    'mask_dir': 'Files/spatial_murine_iCCAvsHCC/segmentation/cleaned_mask',
    'mask_glob': '{fov}_*.tiff',
}

# Optional: override the local workspace root (defaults to ~/.ueler/bia/<accession>/)
local_dir = None

In [ ]:
# Example 2
source = 'S-BIAD2864'   # base path is auto-resolved via the BioStudies API

descriptor = {
    'mode': 'folder',
    'base': 'Files/image_data',            # FOV folders: sample1_fov1, sample1_fov2, …
    'masks': [
        {'dir': 'Files/segmentation_masks', 'name': 'segmentation'},
        {'dir': 'Files/follicle_masks',      'name': 'follicle'},
    ],
}
# Optional: override the local workspace root (defaults to ~/.ueler/bia/<accession>/)
local_dir = None

In [ ]:
# Example 3 — S-BIAD2708 (DCIS): zipped FOVs + per-FOV subfolder masks
source = 'S-BIAD2708'

descriptor = {
    'mode': 'folder',
    'fov_container': 'zip',                 # each FOV is a <FOV>.zip of channel TIFFs
    'base': 'Files/DCIS/image_data',
    'masks': [
        {'dir': 'Files/DCIS/mask_dir', 'per_fov': True},                              # <fov>/ducts_labeled.tiff, myoep_labeled.tiff
        {'dir': 'Files/DCIS/segmentation/deepcell_output', 'match': '{fov}_*.tiff'},  # <fov>_nuclear.tiff, <fov>_whole_cell.tiff
    ],
}
# Optional: override the local workspace root (defaults to ~/.ueler/bia/<accession>/)
local_dir = None

In [ ]:
# Example 4
source = 'S-BIAD1507'   # base path is auto-resolved via the BioStudies API
descriptor= {
    "mode": "folder",
    "fov_container": "zip",
    "base": "Files/TNBC/Spain/image_data/samples",
    "masks": [
        {"dir": "Files/TNBC/Spain/segmentation_data/samples/deepcell_output",
         "match": "{fov}_*.tiff"},   # → labels: nuclear, whole_cell
    ],
}
# Optional: override the local workspace root (defaults to ~/.ueler/bia/<accession>/)
local_dir = None

### Open the viewer

Run the following cell. The first FOV is fetched on demand — only the channels you open are downloaded/streamed.

In [ ]:
import ueler
# Enable interactive widgets backend
%matplotlib widget
viewer = ueler.run_viewer_bia(
    source,
    descriptor=descriptor,
    local_dir=local_dir,
)

### (Optional) Load a cell table

Masks are linked automatically. If you have a cell table for this study (downloaded locally, or
present as a file within the study you have fetched into the workspace `cache/`), attach it the
same way as in `run_ueler.ipynb`:

In [ ]:
# import pandas as pd
# from ueler import load_cell_table
#
# cell_table = pd.read_csv('.../cell_table.csv')  # Replace with your actual cell table path
# load_cell_table(
#     viewer,
#     cell_table=cell_table,
#     auto_display=True,
#     after_plugins=True,
# )